# Knife-edge profile: derivative of `casi_gausiano.txt`

The knife slowly uncovers the beam, so the measured power is the integral of the
beam profile along the knife's travel:

$$P(t) = P_0 + \int_{-\infty}^{t} I(t')\,dt' \quad\Rightarrow\quad I(t) = \frac{dP}{dt}$$

For a Gaussian beam $I \propto e^{-2(x-x_0)^2/w^2}$, so $P(t)$ is an erf.

**Noise:** a plain finite difference `np.diff(P)/np.diff(t)` amplifies the noise, and
the timestamps are irregular (12–69 ms). So we:
1. interpolate onto a uniform 20 ms grid,
2. take the derivative with a **Savitzky–Golay** filter (it fits a local cubic
   polynomial and differentiates that). Several window sizes are compared to check the
   shape isn't a smoothing artifact.

The recording is cut at `t_max`: after that the beam was already fully uncovered.

The x axis is time. To get position, multiply by the knife speed $v$: $x = v\,t$ and $w_x = v\,w_t$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.special import erf

# PM100D log: 2 header lines, then "dd/mm/yyyy hh:mm:ss,fff \t value \t W" (decimal comma)
t, P = [], []
with open('casi_gausiano.txt', encoding='latin-1') as fh:
    for line in fh.read().splitlines()[2:]:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        t.append(datetime.strptime(parts[0].strip(), '%d/%m/%Y %H:%M:%S,%f').timestamp())
        P.append(float(parts[1].replace(',', '.')))
t = np.array(t) - t[0]          # s
P = np.array(P) * 1e3           # mW

# after ~25 s the beam was already fully uncovered and we kept opening the knife:
# the rest is just the laser drifting, so drop it
t_max = 25
keep = t < t_max
t, P = t[keep], P[keep]

print(f'{len(t)} points, dt in [{np.diff(t).min()*1e3:.0f}, {np.diff(t).max()*1e3:.0f}] ms')
print(f'baseline noise ~ {np.std(np.diff(P[t < 3])) / np.sqrt(2) * 1e3:.2f} uW')

## Savitzky–Golay derivative

In [ ]:
dt = 0.02
tu = np.arange(t[0], t[-1], dt)
Pu = np.interp(tu, t, P)

def sg_deriv(win_s, order=3):
    n = int(round(win_s / dt)) | 1   # window length has to be odd
    return savgol_filter(Pu, n, order, deriv=1, delta=dt)

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(t, P, 'k.', ms=2)
ax[0].set_ylabel('P [mW]')
for win in [0.5, 1.0, 2.0]:
    ax[1].plot(tu, sg_deriv(win), lw=1, label=f'SG window {win} s')
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend()
plt.tight_layout()
plt.show()

dPdt = sg_deriv(1.0)
# check: the area under the derivative has to equal the total power step
print(f'area under dP/dt = {np.trapezoid(dPdt, tu):.3f} mW   vs   step = {Pu[-50:].mean() - Pu[:50].mean():.3f} mW')

## Gaussian fits

We compare an erf fit to $P$, a Gaussian fit to $dP/dt$, and a **split Gaussian**
fit to $dP/dt$ (different width on each side). The derivative is clearly asymmetric:
it rises sharply at about 6 s and then has a long tail. That is why the file is *casi* (almost)
Gaussian. Two likely causes: the knife didn't move at constant speed (turning the
screw by hand), or the beam isn't a clean TEM00.

In [ ]:
def erf_model(t, P0, A, t0, w):
    return P0 + A / 2 * (1 + erf(np.sqrt(2) * (t - t0) / w))

def gauss(t, a, t0, w, c):
    return c + a * np.exp(-2 * (t - t0)**2 / w**2)

def split_gauss(t, a, t0, wl, wr, c):
    w = np.where(t < t0, wl, wr)
    return c + a * np.exp(-2 * (t - t0)**2 / w**2)

pe, ce = curve_fit(erf_model, t, P, p0=[1.17, 2.4, 10, 4])
pg, cg = curve_fit(gauss, tu, dPdt, p0=[0.3, 10, 5, 0])
ps, cs = curve_fit(split_gauss, tu, dPdt, p0=[0.3, 10, 3, 6, 0])

for name, p, c, labels in [('erf on P', pe, ce, ['P0', 'A', 't0', 'w']),
                           ('gauss on dP/dt', pg, cg, ['a', 't0', 'w', 'c']),
                           ('split gauss on dP/dt', ps, cs, ['a', 't0', 'w_left', 'w_right', 'c'])]:
    err = np.sqrt(np.diag(c))
    print(name + ':  ' + ',  '.join(f'{l} = {v:.3f} ± {e:.3f}' for l, v, e in zip(labels, p, err)))

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(t, P, 'k.', ms=2, label='data')
ax[0].plot(t, erf_model(t, *pe), 'r-', label='erf fit')
ax[0].set_ylabel('P [mW]')
ax[0].legend()
ax[1].plot(tu, dPdt, 'k-', lw=1, label='dP/dt (SG 1 s)')
ax[1].plot(tu, gauss(tu, *pg), 'r--', label=f'gaussian, w = {pg[2]:.2f} s')
ax[1].plot(tu, split_gauss(tu, *ps), 'b-', label=f'split gaussian, w = {ps[2]:.2f} / {ps[3]:.2f} s')
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend()
plt.tight_layout()
plt.show()